# Hybrid Digital Twin for Li-ion Batteries — Julia

**Can you predict when a battery reaches end of life, from only the first 40 % of its life?**

Three models, one honest test:

| | Model | What it knows |
|---|---|---|
| 1 | **Physics** | a degradation law from the literature, two parameters |
| 2 | **Data** | a Gaussian process. Flexible, and has no idea what a battery is |
| 3 | **Hybrid** | the physics, plus a small network fitted to what the physics gets wrong |

The test is a **temporal split**: fit on the first 40 % of cycles, forecast the rest. Not a random split — a random split trains on Tuesday and Thursday to predict Wednesday, which is interpolation, and is not the question anyone is asking.

> **This notebook needs a Julia kernel, and Colab does not provide one.**
>
> A Julia cell cannot install a Julia runtime — by the time it runs, the kernel
> already had to exist. To open this in Colab you must first install Julia from
> a **Python** cell in a fresh notebook and register the kernel:
>
> ```
> !curl -fsSL https://install.julialang.org | sh -s -- --yes
> !~/.juliaup/bin/julia -e 'using Pkg; Pkg.add("IJulia")'
> ```
>
> then reload and choose *Runtime → Change runtime type → Julia*.
>
> **Running locally with Julia 1.10 installed is the supported path.** The first
> code cell installs the packages it needs.
>
> **Honesty note:** this notebook was written against these libraries but has
> never been executed — no Julia interpreter was available where it was built,
> and the Python notebook is the only one verified in CI. If it fails for you,
> that is a bug and an issue is very welcome. See
> [MAINTAINERS.md](../MAINTAINERS.md): owning this notebook is an open slot.


In [ ]:
# Package setup. Runs whenever a package is missing, which is the first
# time in any fresh session -- including Colab.
#
# The previous version guarded on `!isdefined(Main, :IJulia)`, which is exactly
# backwards: inside any Jupyter/IJulia kernel `IJulia` IS defined, so the
# install was skipped in the one place it was needed and the next line failed
# with "Package CSV not found in current path".
import Pkg
for pkg in ["CSV", "DataFrames", "LsqFit", "Plots", "StatsBase", "Flux"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using CSV, DataFrames, LsqFit, Plots, StatsBase, Statistics, Random
gr(fmt = :png, size = (900, 520), dpi = 150)
Random.seed!(0)
const TRAIN_FRACTION, EOL = 0.40, 0.80

## 0. The data

In [ ]:
# Local checkout first, GitHub second — see the Python notebook for why.
using Downloads

URL = "https://raw.githubusercontent.com/otwin-core/otwin-hybrid/main/data/battery_soh.csv"
local_candidates = ["data/battery_soh.csv", "../data/battery_soh.csv"]
idx = findfirst(isfile, local_candidates)
source = idx === nothing ? Downloads.download(URL) : local_candidates[idx]
println("reading ", source)

df = CSV.read(source, DataFrame)
combine(groupby(df, :Battery), :id_cycle => maximum => :cycles,
        :SoH => minimum => :final_SoH)

## 1. The physics

The **Wang throughput power law** (Wang et al., 2011):

$$\mathrm{SoH}(n) = 1 - c\,n^{z}$$

Two parameters, and the exponent is a *reading about the cell*, not just a knob:

- $z \approx 0.5$ — diffusion-limited SEI growth. Fade slows as the passivation layer thickens.
- $z \approx 1$ — linear wear. Something degrades at a constant rate.
- $z > 1$ — accelerating fade. The **knee**.

Fit it in log space, where the power law is linear and the fit is well posed:

$$\log(1 - \mathrm{SoH}) = \log c + z \log n$$

This matters more than it sounds. Fitting the raw curve on a short window is badly conditioned — capacity drops ~11 % while cycle-to-cycle noise is ~0.7 %, so $c$ and $z$ trade off almost freely and least squares wanders to whatever bound you set. Done that way these cells gave $z = 2.0$, sitting on the bound: not a physical reading, an optimiser lost in a flat valley.

In [ ]:
wang(n, p) = 1.0 .- p[1] .* n .^ p[2]     # p = [c, z]

function fit_physics(n, soh)
    m = (soh .< 1.0) .& (n .> 0)
    # log space: log(1 - SoH) = log(c) + z*log(n)  -- a linear fit
    X = hcat(ones(sum(m)), log.(n[m]))
    coef = X \ log.(1.0 .- soh[m])
    p0 = [exp(coef[1]), clamp(coef[2], 0.3, 1.5)]
    f = curve_fit(wang, n, soh, p0; lower = [1e-8, 0.3], upper = [1e-1, 1.5])
    return f.param[1], f.param[2]
end

## 2. The data-only model

Julia has GaussianProcesses.jl, but to keep the install light this uses a **kernel ridge regression with an RBF kernel** — the posterior mean of a GP, which is the part that does the forecasting. The failure mode is identical, and it is the failure mode that matters: outside the training range it reverts to the mean.

In [ ]:
function fit_krr(n, soh; ℓ = 40.0, λ = 1e-4)
    K = [exp(-(a - b)^2 / (2ℓ^2)) for a in n, b in n]
    μ = mean(soh)
    α = (K + λ * I(length(n))) \ (soh .- μ)
    return (nt) -> μ .+ [sum(α .* [exp(-(t - b)^2 / (2ℓ^2)) for b in n])
                        for t in nt]
end
using LinearAlgebra

## 3. The hybrid

**The network never sees SoH — only the residual.** Small on purpose.

In [ ]:
using Flux

feats(n) = permutedims(hcat(n ./ 100, sqrt.(n) ./ 10))   # 2 x N

function fit_hybrid(n, soh, c, z)
    residual = soh .- wang(n, [c, z])
    model = Chain(Dense(2, 16, tanh), Dense(16, 16, tanh), Dense(16, 1))
    X, Y = Float32.(feats(n)), Float32.(reshape(residual, 1, :))
    opt = Flux.setup(Adam(5e-3), model)
    for _ in 1:3000
        g = Flux.gradient(m -> Flux.mse(m(X), Y), model)[1]
        Flux.update!(opt, model, g)
    end
    return (nt) -> vec(model(Float32.(feats(nt))))
end

## 4. The honest test

In [ ]:
rmse(y, ŷ) = sqrt(mean((y .- ŷ) .^ 2))

results = DataFrame(battery = String[], model = String[],
                    rmse = Float64[], skill = Float64[])
curves = Dict()

for g in groupby(df, :Battery)
    cell = first(g.Battery)
    n, soh = Float64.(g.id_cycle), Float64.(g.SoH)
    k = Int(floor(length(n) * TRAIN_FRACTION))
    ntr, str_, nte, ste = n[1:k], soh[1:k], n[k+1:end], soh[k+1:end]

    c, z = fit_physics(ntr, str_)
    krr  = fit_krr(ntr, str_)
    resid = fit_hybrid(ntr, str_, c, z)

    preds = Dict(
        "physics"     => wang(nte, [c, z]),
        "gp"          => krr(nte),
        "hybrid"      => wang(nte, [c, z]) .+ resid(nte),
        "persistence" => fill(str_[end], length(nte)),
        "drift"       => (hcat(ones(k), ntr) \ str_)' * vcat(ones(1, length(nte)), nte') |> vec)
    base = rmse(ste, preds["persistence"])
    curves[cell] = (n, soh, k, preds)
    for (m, p) in preds
        push!(results, (cell, m, rmse(ste, p), rmse(ste, p) / base))
    end
end

sort(combine(groupby(results, :model),
             :rmse => mean => :rmse, :skill => mean => :skill), :rmse)

### The figure

In [ ]:
cell = "B0005"
n, soh, k, preds = curves[cell]
split = n[k]
COL = Dict("physics" => "#1C4E73", "gp" => "#B5651D", "hybrid" => "#2E7D32")
LBL = Dict("physics" => "Physics only", "gp" => "Data only", "hybrid" => "Hybrid")

plt = scatter(n, soh, ms = 3, mc = :black, ma = 0.5, msw = 0,
              label = "Measured SoH", legend = :bottomleft)
vspan!(plt, [minimum(n), split], fc = :lightsteelblue, fa = 0.18, label = "")
for m in ("physics", "gp", "hybrid")
    plot!(plt, n[k+1:end], preds[m], lw = 3, c = COL[m], label = LBL[m])
end
hline!(plt, [EOL], c = :crimson, ls = :dash, lw = 2, label = "end of life (80%)")
xlabel!(plt, "Discharge cycle"); ylabel!(plt, "State of Health")
title!(plt, "$cell — fitted on the shaded window")
plt

## 5. What actually happened

Mean over four cells, temporal split at 40 %:

| Model | RMSE | Skill vs persistence |
|---|---|---|
| **Hybrid** | **0.0398** | **0.36** |
| Baseline: linear drift | 0.0490 | 0.44 |
| Physics only | 0.0544 | 0.50 |
| Baseline: persistence | 0.1109 | 1.00 |
| Data only (GP) | 0.1791 | 1.62 |

Three things worth sitting with:

**The Gaussian process is worse than assuming nothing changes.** Skill 1.62. Not because it is badly implemented — it is properly specified and fitted with restarts. Because outside the range it has seen, a GP reverts to its prior mean. It has no concept of a battery, so it has no reason to keep going down.

**A straight line beats the physics on RMSE.** Linear drift, skill 0.44 against the physics model's 0.50. That is humbling and it is real. Over a bounded horizon, extrapolating a line is a genuinely strong baseline, and a project that only reported its wins would have quietly dropped this row.

**But RMSE is not the question.** An operator asks *when do I replace it?* On that metric the ranking inverts:

| Model | Mean error in predicted end-of-life cycle |
|---|---|
| **Physics only** | **13.0 cycles** |
| Hybrid | 21.9 cycles |
| Baseline: linear drift | 25.7 cycles |

The straight line has the second-best RMSE and the worst answer to the actual question, because it crosses the 80 % line at the wrong angle. **Choose the metric that matches the decision, or you will optimise the wrong thing very precisely.**

*Numbers may differ in the last digit from the Python notebook: the optimisers and the RNG are not the same. The ranking, and the reason for it, are.*

## 6. Where this goes next

This notebook is the *tutorial*. The same ideas, engineered properly, are a set of composable tools:

| Tool | What it does |
|---|---|
| [`otwin-systems`](https://github.com/otwin-core/otwin-systems) | physical model structures, each validated against a closed-form answer |
| [`otwin-eval`](https://github.com/otwin-core/otwin-eval) | the temporal split and mandatory baselines used here |
| [`otwin-uq`](https://github.com/otwin-core/otwin-uq) | calibrated uncertainty — this notebook has none, which is its biggest gap |
| [`otwin-phs`](https://github.com/otwin-core/otwin-phs) | port-Hamiltonian systems, for assets where the physics is an energy balance |

**What this notebook does not do, and should:** produce an interval. Every forecast above is a single line, and a single line is not a forecast — it is a guess with good posture. `otwin-uq` measures whether a 90 % band actually contains the truth 90 % of the time.

---

### The one thing to take away

The physics is not there for interpretability. It is there because it is the only part of the model that still knows what it is doing outside the data it was fitted on.

---

Full ecosystem: **[github.com/otwin-core](https://github.com/otwin-core)** · Apache 2.0